<a href="https://colab.research.google.com/github/Paul412399/PROIECT-PI/blob/main/Vaci_Schiopatat_Detectie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### Celula 1 — Montează Drive și setează căile

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive")

DATA_DIR = DRIVE_DIR / "CattleLameness" / "Data"

POSE_WEIGHTS = DRIVE_DIR / "cow_pose_runs" / "yolov8m_nwafu20_ft1" / "weights" / "best.pt"

RUN_DIR = DRIVE_DIR / "cow_lameness_pose_run"
FEATURE_DIR = RUN_DIR / "features"

RUN_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Nu găsesc folderul cu date: {DATA_DIR}"
assert (DATA_DIR / "Normal").exists(), f"Lipsește: {DATA_DIR / 'Normal'}"
assert (DATA_DIR / "Lame").exists(), f"Lipsește: {DATA_DIR / 'Lame'}"
assert POSE_WEIGHTS.exists(), f"Nu găsesc modelul pose: {POSE_WEIGHTS}"

print("Date:", DATA_DIR)
print("Model pose:", POSE_WEIGHTS)
print("Rezultate/checkpointuri:", RUN_DIR)

In [ ]:
### Celula 2 — Instalează pachetele

!pip -q install ultralytics opencv-python-headless scikit-learn

In [ ]:
### Celula 3 — Verifică GPU-ul și încarcă modelul pose


import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch nu vede GPU-ul. În Colab selectează "
        "Runtime > Change runtime type > GPU."
    )

device = 0
print("GPU:", torch.cuda.get_device_name(0))

pose_model = YOLO(str(POSE_WEIGHTS))

kpt_shape = pose_model.model.yaml.get("kpt_shape")
print("Forma keypoint-urilor:", kpt_shape)

assert kpt_shape is not None, "Modelul încărcat nu pare să fie un model pose."
NUM_KEYPOINTS = int(kpt_shape[0])

In [ ]:
### Celula 4 — Găsește clipurile și atribuie etichetele


import pandas as pd

CLASS_TO_ID = {
    "Normal": 0,
    "Lame": 1,
}

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}

rows = []

for class_name, label in CLASS_TO_ID.items():
    class_dir = DATA_DIR / class_name

    for path in sorted(class_dir.iterdir()):
        if path.suffix.lower() in VIDEO_EXTS:
            rows.append({
                "path": str(path),
                "video_id": f"{class_name}_{path.stem}",
                "class_name": class_name,from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

folder = Path("/content/runs/pose/predict")
imagini = sorted(
    p for p in folder.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

if not imagini:
    print(f"Nu am găsit imagini în {folder}")
else:
    coloane = 4
    randuri = math.ceil(len(imagini) / coloane)
    fig, axe = plt.subplots(randuri, coloane, figsize=(16, 4 * randuri))
    axe = [axe] if randuri * coloane == 1 else axe.flatten()

    for ax, cale in zip(axe, imagini):
        ax.imshow(Image.open(cale))
        ax.set_title(cale.name)
        ax.axis("off")

    for ax in axe[len(imagini):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

                "label": label,
            })

videos = pd.DataFrame(rows)

print(videos.groupby(["class_name", "label"]).size())
print("Total clipuri:", len(videos))

assert len(videos) > 0, "Nu am găsit fișiere video."

class_name  label
Lame        1        25
Normal      0        25
dtype: int64
Total clipuri: 50


In [ ]:
### Celula 5 — Funcții pentru extragerea articulațiilor din video


import cv2
import numpy as np

N_FRAMES = 32
IMGSZ = 960
POSE_CONF = 0.05

def extract_video_features(video_path, n_frames=N_FRAMES):

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise ValueError(f"Nu pot deschide clipul: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f"Clipul nu are cadre lizibile: {video_path}")

    frame_indices = np.linspace(0, total_frames - 1, n_frames).astype(int)
    sequence = np.zeros((n_frames, NUM_KEYPOINTS, 3), dtype=np.float32)
    detected = np.zeros(n_frames, dtype=bool)

    for t, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()

        if not ok:
            continue

        result = pose_model.predict(
            source=frame,
            imgsz=IMGSZ,
            conf=POSE_CONF,
            device=device,
            verbose=False,
        )[0]

        if result.keypoints is None or result.keypoints.data is None:
            continue

        all_kpts = result.keypoints.data.detach().cpu().numpy()

        if len(all_kpts) == 0:
            continue

        if result.boxes is not None and len(result.boxes) == len(all_kpts):
            boxes = result.boxes.xyxy.detach().cpu().numpy()
            areas = (
                (boxes[:, 2] - boxes[:, 0])
                * (boxes[:, 3] - boxes[:, 1])
            )
            chosen = int(np.argmax(areas))
        else:
            chosen = 0

        points = all_kpts[chosen]
        h, w = frame.shape[:2]

        n_points = min(NUM_KEYPOINTS, points.shape[0])
        sequence[t, :n_points, 0] = points[:n_points, 0] / max(w, 1)
        sequence[t, :n_points, 1] = points[:n_points, 1] / max(h, 1)
        sequence[t, :n_points, 2] = points[:n_points, 2]
        detected[t] = True

    cap.release()

    if not detected.any():
        raise ValueError(f"Pose modelul nu a detectat vaca în niciun cadru: {video_path}")

    valid_indices = np.flatnonzero(detected)
    for t in np.flatnonzero(~detected):
        nearest = valid_indices[np.argmin(np.abs(valid_indices - t))]
        sequence[t] = sequence[nearest]

    return sequence


In [ ]:
## Celula 5.5 - Verificarea punctelor cu ochiul liber

import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_pose_examples(class_name, n_samples=6):
    row = videos[videos["class_name"] == class_name].iloc[0]
    video_path = row["path"]

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if not cap.isOpened() or total_frames <= 0:
        cap.release()
        print("Nu pot deschide clipul:", video_path)
        return

    indices = np.linspace(0, total_frames - 1, n_samples).astype(int)
    fig, axes = plt.subplots(1, n_samples, figsize=(4 * n_samples, 4))
    if n_samples == 1:
        axes = [axes]

    print(f"\nClasa: {class_name} | Clip: {video_path}")

    for ax, frame_idx in zip(axes, indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

folder = Path("/content/runs/pose/predict")
imagini = sorted(
    p for p in folder.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

if not imagini:
    print(f"Nu am găsit imagini în {folder}")
else:
    coloane = 4
    randuri = math.ceil(len(imagini) / coloane)
    fig, axe = plt.subplots(randuri, coloane, figsize=(16, 4 * randuri))
    axe = [axe] if randuri * coloane == 1 else axe.flatten()

    for ax, cale in zip(axe, imagini):
        ax.imshow(Image.open(cale))
        ax.set_title(cale.name)
        ax.axis("off")

    for ax in axe[len(imagini):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

        if not ok:
            ax.set_title(f"Cadru {frame_idx}: nu s-a putut citi")
            ax.axis("off")
            continue

        result = pose_model.predict(
            source=frame,
            imgsz=IMGSZ,
            conf=POSE_CONF,
            device=device,
            verbose=False,
        )[0]

        annotated = result.plot()

        if result.keypoints is not None and result.keypoints.data is not None:
            kpts = result.keypoints.data.detach().cpu().numpy()
            n_cows = len(kpts)

            if n_cows > 0:
                mean_conf = float(kpts[:, :, 2].mean())
                title = f"cadru {frame_idx} | vaci: {n_cows} | conf: {mean_conf:.2f}"
            else:
                title = f"cadru {frame_idx} | fără keypoint-uri"
        else:
            title = f"cadru {frame_idx} | fără detecție pose"

        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    cap.release()
    plt.tight_layout()
    plt.show()

show_pose_examples("Normal")
show_pose_examples("Lame")

In [ ]:
### Celula 6 — Extrage features și salvează progresul în Drive


from tqdm.auto import tqdm

failed = []

for row in tqdm(videos.itertuples(index=False), total=len(videos)):
    output_path = FEATURE_DIR / f"{row.video_id}.npy"

    if output_path.exists():
        continue

    try:
        features = extract_video_features(row.path)
        np.save(output_path, features)
    except Exception as e:
        failed.append((row.video_id, str(e)))
        print(f"Eroare la {row.video_id}: {e}")

print("Features salvate:", len(list(FEATURE_DIR.glob("*.npy"))))
print("Clipuri eșuate:", len(failed))

if failed:
    print("Primele erori:", failed[:5])


In [ ]:
### Celula 7 — Împarte datele pe clipuri


from sklearn.model_selection import train_test_split

X_list, y_list, ids = [], [], []

for row in videos.itertuples(index=False):
    feature_path = FEATURE_DIR / f"{row.video_id}.npy"

    if not feature_path.exists():
        continue

    X_list.append(np.load(feature_path))
    y_list.append(row.label)
    ids.append(row.video_id)

X = np.asarray(X_list, dtype=np.float32)
y = np.asarray(y_list, dtype=np.int64)
ids = np.asarray(ids)

print("X shape:", X.shape)
print("Etichete:", np.bincount(y))

assert len(X) >= 10, "Sunt prea puține clipuri procesate pentru împărțire train/test."

X_trainval, X_test, y_trainval, y_test, ids_trainval, ids_test = train_test_split(
    X, y, ids,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_trainval, y_trainval, ids_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval,
)

np.savez(
    RUN_DIR / "split_ids.npz",
    train=ids_train,
    validation=ids_val,
    test=ids_test,
)

print("Train:", len(y_train))
print("Validation:", len(y_val))
print("Test:", len(y_test))

In [ ]:
### Celula 8 — Definește și antrenează clasificatorul Normal/Șchiop


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)

n_frames = X_train.shape[1]
input_size = X_train.shape[2] * X_train.shape[3]

X_train_seq = X_train.reshape(len(X_train), n_frames, input_size)
X_val_seq = X_val.reshape(len(X_val), n_frames, input_size)

def build_classifier(input_shape):
    inputs = keras.Input(shape=input_shape)

    x = layers.Conv1D(128, kernel_size=5, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model

BEST_MODEL_PATH = RUN_DIR / "best_classifier.keras"
LAST_MODEL_PATH = RUN_DIR / "last_classifier.keras"
BACKUP_DIR = RUN_DIR / "training_backup"

if LAST_MODEL_PATH.exists():
    print("Reiau din ultimul checkpoint.")
    classifier = keras.models.load_model(LAST_MODEL_PATH)
else:
    classifier = build_classifier((n_frames, input_size))

classes, counts = np.unique(y_train, return_counts=True)
class_weight = {
    int(c): len(y_train) / (len(classes) * count)
    for c, count in zip(classes, counts)
}
print("Class weights:", class_weight)

callbacks = [
    keras.callbacks.BackupAndRestore(
        backup_dir=str(BACKUP_DIR),
        save_freq="epoch",
        delete_checkpoint=False,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(LAST_MODEL_PATH),
        save_best_only=False,
        save_freq="epoch",
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_auc",
        mode="max",
        save_best_only=True,
    ),
    keras.callbacks.CSVLogger(
        filename=str(RUN_DIR / "classifier_training_log.csv"),
        append=True,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=15,
        restore_best_weights=True,
    ),
]

history = classifier.fit(
    X_train_seq,
    y_train,
    validation_data=(X_val_seq, y_val),
    epochs=100,
    batch_size=8,
    class_weight=class_weight,
    callbacks=callbacks,
)


In [ ]:
### Celula 9 — Evaluează modelul pe clipurile ținute deoparte

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

assert BEST_MODEL_PATH.exists(), "Nu găsesc modelul cel mai bun. Rulează celula de antrenare."

best_classifier = keras.models.load_model(BEST_MODEL_PATH)
X_test_seq = X_test.reshape(len(X_test), n_frames, input_size)

probabilities = best_classifier.predict(X_test_seq, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("Confusion matrix (rânduri=real, coloane=prezis):")
print(confusion_matrix(y_test, predictions, labels=[0, 1]))

print(classification_report(
    y_test,
    predictions,
    labels=[0, 1],
    target_names=["Normal (0)", "Schiop (1)"],
    zero_division=0,
))

if len(np.unique(y_test)) == 2:
    print("Test ROC-AUC:", roc_auc_score(y_test, probabilities))

print("\nClipuri test și predicții:")
for video_id, true_label, pred, prob in zip(ids_test, y_test, predictions, probabilities):
    print(
        f"{video_id}: real={true_label}, prezis={pred}, "
        f"prob_schiop={prob:.3f}"
    )

Confusion matrix (rânduri=real, coloane=prezis):
[[2 3]
 [4 1]]
              precision    recall  f1-score   support

  Normal (0)       0.33      0.40      0.36         5
  Schiop (1)       0.25      0.20      0.22         5

    accuracy                           0.30        10
   macro avg       0.29      0.30      0.29        10
weighted avg       0.29      0.30      0.29        10

Test ROC-AUC: 0.19999999999999998

Clipuri test și predicții:
Normal_N (19): real=0, prezis=1, prob_schiop=0.554
Normal_N (15): real=0, prezis=1, prob_schiop=0.594
Normal_N (22): real=0, prezis=0, prob_schiop=0.497
Normal_N (4): real=0, prezis=0, prob_schiop=0.285
Lame_L (18): real=1, prezis=0, prob_schiop=0.229
Lame_L (9): real=1, prezis=0, prob_schiop=0.365
Lame_L (24): real=1, prezis=1, prob_schiop=0.578
Lame_L (2): real=1, prezis=0, prob_schiop=0.370
Normal_N (16): real=0, prezis=1, prob_schiop=0.756
Lame_L (6): real=1, prezis=0, prob_schiop=0.224
